In [4]:
import pandas as pd
import re

# 📂 Dataset paths
files = [
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Assassin.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Nazario_5.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Nigerian_5.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Enron.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/CEAS-08.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Ling.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/TREC-07.csv"
]

# 🔍 Normalize obfuscation
def normalize_text(text):
    text = str(text)
    
    text = text.replace("hxxp", "http")
    text = text.replace("hxxps", "https")
    text = text.replace("[.]", ".")
    text = text.replace("(dot)", ".")
    text = text.replace(" dot ", ".")
    
    return text

# 🔗 Extract URLs
def extract_urls(text):
    text = normalize_text(text)
    
    url_pattern = r'(https?://[^\s]+|www\.[^\s]+)'
    return re.findall(url_pattern, text)

# 🧹 Clean URLs
def clean_url(url):
    url = url.strip().lower()
    
    # Remove trailing junk
    url = re.sub(r'[^\w:/\.\-]', '', url)
    
    return url

# 🔧 Fix missing protocol
def fix_url(url):
    if url.startswith("www."):
        return "http://" + url
    return url

# 🚀 MAIN PIPELINE
url_rows = []

for file in files:
    print(f"\n📂 Processing: {file.split('/')[-1]}")
    
    df = pd.read_csv(file)
    
    # Combine subject + body
    df['text'] = df['subject'].fillna('') + " " + df['body'].fillna('')
    
    # Ensure label numeric
    df['label'] = pd.to_numeric(df['label'], errors='coerce')
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    
    for _, row in df.iterrows():
        text = row['text']
        label = row['label']
        
        urls = extract_urls(text)
        
        for url in urls:
            cleaned = clean_url(url)
            fixed = fix_url(cleaned)
            
            if len(fixed) > 10:  # remove very short/noisy URLs
                url_rows.append({
                    "url": fixed,
                    "label": label,
                    "source": file.split('/')[-1]  # useful for analysis
                })

# 📊 Create DataFrame
url_df = pd.DataFrame(url_rows)

print("\nTotal extracted URLs:", len(url_df))

# ❗ Remove duplicates
url_df = url_df.drop_duplicates(subset=['url']).reset_index(drop=True)

print("Unique URLs:", len(url_df))

# 📊 Label distribution
print("\nLabel distribution:")
print(url_df['label'].value_counts())

# 💾 Save dataset
output_path = "/kaggle/working/phishing_url_dataset.csv"
url_df.to_csv(output_path, index=False)

print(f"\n✅ Dataset saved to: {output_path}")


📂 Processing: Assassin.csv

📂 Processing: Nazario_5.csv

📂 Processing: Nigerian_5.csv

📂 Processing: Enron.csv

📂 Processing: CEAS-08.csv

📂 Processing: Ling.csv

📂 Processing: TREC-07.csv

Total extracted URLs: 252441
Unique URLs: 73218

Label distribution:
label
0    52603
1    20615
Name: count, dtype: int64

✅ Dataset saved to: /kaggle/working/phishing_url_dataset.csv


In [6]:
import pandas as pd
import re

# 📂 Dataset paths
files = [
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Assassin.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Nazario_5.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Nigerian_5.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Enron.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/CEAS-08.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/Ling.csv",
    "/kaggle/input/datasets/priyadharsana/phishunter-datasets/TREC-07.csv"
]

# 🔍 Normalize obfuscation
def normalize_text(text):
    text = str(text)
    
    text = text.replace("hxxp", "http")
    text = text.replace("hxxps", "https")
    text = text.replace("(dot)", ".")
    text = text.replace("[.]", ".")
    
    return text

# 🧹 Transformer-friendly cleaning
def clean_email_text(text):
    text = normalize_text(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove HTML
    text = re.sub(r'<.*?>', ' ', text)
    
    # 🔥 Replace URLs (no leakage)
    text = re.sub(r'http\S+|www\S+', ' LINK ', text)
    
    # Replace emails
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)
    
    # Normalize money
    text = re.sub(r'\$\s?\d+', ' MONEY ', text)
    
    # Remove non-ASCII junk
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    
    # Normalize spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# 🚀 MAIN PIPELINE
all_dfs = []

for file in files:
    print(f"\n📂 Processing: {file.split('/')[-1]}")
    
    df = pd.read_csv(file)
    
    # Fill missing safely
    df['subject'] = df['subject'].fillna('')
    df['body'] = df['body'].fillna('')
    
    # Ensure label numeric
    df['label'] = pd.to_numeric(df['label'], errors='coerce')
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    
    # Remove very short emails (combined length check)
    df = df[(df['subject'] + df['body']).str.split().str.len() > 5]
    
    # 🧹 Clean separately
    df['subject_clean'] = df['subject'].apply(clean_email_text)
    df['body_clean'] = df['body'].apply(clean_email_text)
    
    # Keep structured columns
    df = df[['subject', 'body', 'subject_clean', 'body_clean', 'label']]
    
    # Add source
    df['source'] = file.split('/')[-1]
    
    print("Shape:", df.shape)
    print(df['label'].value_counts())
    
    all_dfs.append(df)

# 🔗 Combine all datasets
master_df = pd.concat(all_dfs, ignore_index=True)

# 🔀 Shuffle
master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n✅ Final Master Dataset Shape:", master_df.shape)
print(master_df.head())


📂 Processing: Assassin.csv
Shape: (5804, 6)
label
0    4091
1    1713
Name: count, dtype: int64

📂 Processing: Nazario_5.csv
Shape: (3059, 6)
label
1    1560
0    1499
Name: count, dtype: int64

📂 Processing: Nigerian_5.csv
Shape: (6326, 6)
label
1    3331
0    2995
Name: count, dtype: int64

📂 Processing: Enron.csv
Shape: (29714, 6)
label
0    15767
1    13947
Name: count, dtype: int64

📂 Processing: CEAS-08.csv
Shape: (38529, 6)
label
1    21221
0    17308
Name: count, dtype: int64

📂 Processing: Ling.csv
Shape: (2859, 6)
label
0    2401
1     458
Name: count, dtype: int64

📂 Processing: TREC-07.csv
Shape: (53715, 6)
label
1    29357
0    24358
Name: count, dtype: int64

✅ Final Master Dataset Shape: (140006, 6)
                                             subject  \
0                How interesting is your love life?    
1  [Ip-health] CQ Weekly - The Shadowy Drug Lobby...   
2                Re: [ILUG-Social] Online Bookstores   
3                                                Re

In [3]:
master_df['model_input'] = (
    master_df['subject_clean'] + " " + master_df['body_clean']
)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    master_df['model_input'],
    master_df['label'],
    test_size=0.2,
    random_state=42,
    stratify=master_df['label']
)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# Vectorization
vectorizer = TfidfVectorizer(max_features=10000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Model
svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)

# Prediction
y_pred_svm = svm_model.predict(X_test_vec)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.9859652881937004
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     13684
           1       0.99      0.99      0.99     14318

    accuracy                           0.99     28002
   macro avg       0.99      0.99      0.99     28002
weighted avg       0.99      0.99      0.99     28002



In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 🔹 Tokenization
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# 🔹 Padding
max_len = 200
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

# 🔥 MODEL (Improved)
lstm_model = Sequential([
    Embedding(input_dim=20000, output_dim=128, input_length=max_len),
    
    LSTM(64, return_sequences=True),   # 🔥 stacked LSTM
    Dropout(0.3),
    
    LSTM(32),                          # 🔥 second layer
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# 🔥 Early stopping (important)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# 🔥 TRAIN (GPU optimized)
history = lstm_model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=128,   # 🔥 larger batch for GPU
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# 🔹 Evaluate
loss, acc = lstm_model.evaluate(X_test_pad, y_test)
print("LSTM Accuracy:", acc)

2026-04-04 04:10:30.611743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775275831.065357      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775275831.180709      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775275832.237101      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775275832.237132      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775275832.237135      55 computation_placer.cc:177] computation placer alr

Epoch 1/5


I0000 00:00:1775275895.594277      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775275895.597226      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1775275901.088547     129 cuda_dnn.cc:529] Loaded cuDNN version 91002


788/788 ━━━━━━━━━━━━━━━━━━━━ 23s 19ms/step - accuracy: 0.7471 - loss: 0.5234 - val_accuracy: 0.8621 - val_loss: 0.3409
Epoch 2/5
788/788 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9244 - loss: 0.2276 - val_accuracy: 0.9666 - val_loss: 0.1009
Epoch 3/5
788/788 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9731 - loss: 0.0911 - val_accuracy: 0.9771 - val_loss: 0.0730
Epoch 4/5
788/788 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9813 - loss: 0.0654 - val_accuracy: 0.9742 - val_loss: 0.1020
Epoch 5/5
788/788 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.9853 - loss: 0.0531 - val_accuracy: 0.9822 - val_loss: 0.0656
876/876 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9842 - loss: 0.0579
LSTM Accuracy: 0.9841082692146301


In [14]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Convert to HF dataset
train_dataset = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Model
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)
model.to(device)

def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# Training args
training_args = TrainingArguments(
    output_dir="./results",
    # 🔥 GPU optimization
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    fp16=True,   # 🔥 VERY IMPORTANT
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train
trainer.train()

# Evaluate
trainer.evaluate()

Map:   0%|          | 0/112004 [00:00<?, ? examples/s]

Map:   0%|          | 0/28002 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using: cuda


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.055068,0.051713,0.994715,0.994823,0.996496,0.993155
2,0.021198,0.043755,0.996286,0.996364,0.997550,0.995181


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.04375458136200905,
 'eval_accuracy': 0.9962859795728877,
 'eval_f1': 0.9963638906370184,
 'eval_precision': 0.9975497059647158,
 'eval_recall': 0.9951808911859198,
 'eval_runtime': 246.6727,
 'eval_samples_per_second': 113.519,
 'eval_steps_per_second': 3.551,
 'epoch': 2.0}

In [35]:
sample_texts = [
    "please verify your account",  # subtle phishing
    "meeting rescheduled to tomorrow",  # legit
    "click here to update your password !!!",  # phishing
    "free free free offer now!!!",  # spam
]

inputs = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True, max_length=256)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)
preds = torch.argmax(outputs.logits, dim=1)

print(preds)

tensor([1, 0, 1, 1], device='cuda:0')


In [8]:
import joblib

# Save vectorizer
joblib.dump(vectorizer, "/kaggle/working/tfidf_vectorizer.pkl")

# Save SVM model
joblib.dump(svm_model, "/kaggle/working/svm_model.pkl")

print("✅ SVM model saved")

NameError: name 'vectorizer' is not defined

In [30]:
lstm_model.save("/kaggle/working/lstm_model.h5")

In [31]:
import pickle

with open("/kaggle/working/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Tokenizer saved")

✅ Tokenizer saved


In [32]:
trainer.save_model("/kaggle/working/roberta_model")

# Save tokenizer
tokenizer.save_pretrained("/kaggle/working/roberta_model")

print("✅ RoBERTa model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RoBERTa model saved


In [33]:
import shutil

shutil.make_archive(
    "/kaggle/working/roberta_model",  # zip name
    'zip',
    "/kaggle/working/roberta_model"   # folder to zip
)

print("✅ RoBERTa model zipped")

✅ RoBERTa model zipped


In [7]:
# ============================
# 🔥 PHISHUNTER LAYER 3 TRAINING (FINAL CLEAN)
# ============================

import pandas as pd
import numpy as np
import re
import torch
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib
import json

# ----------------------------
# STEP 0: DEVICE
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------------------
# STEP 1: TEXT
# ----------------------------
master_df = master_df.copy()
master_df = master_df.loc[:, ~master_df.columns.duplicated()]

master_df['text'] = master_df['subject_clean'].fillna('') + " " + master_df['body_clean'].fillna('')
master_df['text'] = master_df['text'].str[:1000]

# ----------------------------
# STEP 2: LEXICONS
# ----------------------------
urgency_words = ["urgent","immediately","now","asap","act now","limited time","deadline"]
fear_words = ["suspended","blocked","legal action","unauthorized","breach","compromised"]
authority_words = ["ceo","admin","bank","official","support team","security department"]
reward_words = ["won","prize","reward","free","gift","bonus","offer"]

pressure_patterns = [r"you need to", r"you must", r"act now", r"do it now"]
urgency_patterns = [r"before it'?s too late", r"limited time", r"last chance", r"only today"]
suggestive_patterns = [r"this is your chance", r"you deserve", r"don’t miss", r"you should try"]
soft_patterns = [r"improve your", r"enhance your", r"what you need", r"interesting", r"better life"]

# ----------------------------
# STEP 3: FEATURE EXTRACTION
# ----------------------------
def count_matches(text, words):
    return sum(len(re.findall(r'\b' + re.escape(word) + r'\b', text)) for word in words)

def pattern_score(text, patterns):
    return sum(1 for p in patterns if re.search(p, text))

def extract_features(text):
    if not isinstance(text, str):
        text = ""
        
    t = text.lower()
    wc = len(t.split()) + 1

    urgency = count_matches(t, urgency_words) / wc
    fear = count_matches(t, fear_words) / wc
    authority = count_matches(t, authority_words) / wc
    reward = count_matches(t, reward_words) / wc

    pressure = pattern_score(t, pressure_patterns)
    urgency_pattern = pattern_score(t, urgency_patterns)
    suggestive = pattern_score(t, suggestive_patterns)
    soft = pattern_score(t, soft_patterns)

    caps = sum(1 for c in text if c.isupper()) / (len(text) + 1)
    excl = text.count("!") / (len(text) + 1)

    return [
        urgency, fear, authority, reward,
        pressure, urgency_pattern, suggestive,
        soft,
        caps, excl
    ]

# Clean old columns if rerun
cols = [
    "urgency","fear","authority","reward",
    "pressure","urgency_pattern","suggestive",
    "soft","caps_ratio","exclamations"
]

for col in cols:
    if col in master_df.columns:
        master_df.drop(columns=[col], inplace=True)

# Apply features
features = master_df['text'].apply(extract_features)
feature_df = pd.DataFrame(features.tolist(), columns=cols)

master_df = pd.concat([master_df, feature_df], axis=1)

# ----------------------------
# STEP 4: LOAD BERT
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = AutoModel.from_pretrained('distilbert-base-uncased').to(device)

# ----------------------------
# STEP 5: BATCH EMBEDDINGS
# ----------------------------
def get_embeddings_batch(texts, batch_size=32):
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size].tolist()
        
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128
        )
        
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = bert_model(**inputs)
        
        cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.extend(cls)
    
    return np.array(embeddings)

# ----------------------------
# STEP 6: SAMPLE
# ----------------------------
sample_df = master_df.sample(20000, random_state=42)

bert_embeddings = get_embeddings_batch(sample_df['text'])
bert_df = pd.DataFrame(bert_embeddings)
bert_df.columns = [f"bert_{i}" for i in range(bert_df.shape[1])]

# ----------------------------
# STEP 7: FUSION
# ----------------------------
X_features = sample_df[cols].reset_index(drop=True)

X = pd.concat([bert_df, X_features], axis=1)
X.columns = X.columns.astype(str)

y = sample_df['label'].values

# ----------------------------
# STEP 8: MODEL
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=120, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# ----------------------------
# STEP 9: MANIPULATION SCORE
# ----------------------------
master_df['manipulation_score'] = (
    0.6 * (
        master_df['urgency'] +
        master_df['fear'] +
        master_df['authority'] +
        master_df['reward']
    )
    +
    0.25 * (
        2 * master_df['pressure'] +
        1.5 * master_df['urgency_pattern'] +
        master_df['suggestive']
    )
    +
    0.1 * master_df['soft']
    +
    0.05 * (
        master_df['caps_ratio'] +
        master_df['exclamations']
    )
)

# ----------------------------
# STEP 10: EXPLAINABILITY
# ----------------------------
def explain(text):
    vals = extract_features(text)
    labels = [
        "urgency","fear","authority","reward",
        "pressure","urgency_pattern","suggestive",
        "implicit_persuasion"
    ]
    return [labels[i] for i in range(len(labels)) if vals[i] > 0]

# ----------------------------
# TEST
# ----------------------------
sample = master_df.iloc[0]

print("\n🔍 SAMPLE ANALYSIS")
print("TEXT:\n", sample['text'][:300])
print("\nManipulation Score:", sample['manipulation_score'])
print("Detected Signals:", explain(sample['text']))

Using device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 625/625 [01:04<00:00,  9.72it/s]



📊 Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.94      0.95      1970
           1       0.94      0.97      0.96      2030

    accuracy                           0.95      4000
   macro avg       0.95      0.95      0.95      4000
weighted avg       0.95      0.95      0.95      4000


🔍 SAMPLE ANALYSIS
TEXT:
 how interesting is your love life? if good health is what you really need, then its time to visit canadian chemists. LINK

Manipulation Score: 0.10163934426229508
Detected Signals: ['implicit_persuasion']


In [8]:
import joblib

# Save classifier
joblib.dump(clf, "layer3_rf_model.pkl")

['layer3_rf_model.pkl']

In [9]:
# Save locally
bert_model.save_pretrained("bert_model")
tokenizer.save_pretrained("bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_model/tokenizer_config.json', 'bert_model/tokenizer.json')

In [10]:
config = {
    "urgency_words": urgency_words,
    "fear_words": fear_words,
    "authority_words": authority_words,
    "reward_words": reward_words,
    "pressure_patterns": pressure_patterns,
    "urgency_patterns": urgency_patterns,
    "suggestive_patterns": suggestive_patterns,
    "soft_patterns": soft_patterns
}

with open("layer3_config.json", "w") as f:
    json.dump(config, f)

In [11]:
import zipfile
import os

zip_filename = "phishunter_layer3.zip"

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for root, dirs, files in os.walk("bert_model"):
        for file in files:
            filepath = os.path.join(root, file)
            zipf.write(filepath)

print("✅ Layer 3 exported as ZIP")

✅ Layer 3 exported as ZIP
